In [ ]:
# Import libraries
import pandas as pd
import os
from pathlib import Path
import networkx as nx
import numpy as np
import pickle

from utils.graph import draw_graph, create_graph, create_multi_graph, draw_multi_graph

In [ ]:
train_dir = r"data/eICU_first24h_ts1h/train"
Path(train_dir).mkdir(exist_ok=True, parents=True)

In [ ]:
# load data
demo_df = pd.read_csv(os.path.join(train_dir, "demographics.csv"), sep=',', dtype={'icd9_code': str})
icd9_codes = demo_df["icd9_code"].unique()

graph_names = ["comorbidity_graph", "usability_graph", "drug_graph"]

## Build Integreted Graph (sum)
Create a graph with the edge weight of sum of the previous three graphs

In [ ]:
# create all graph for each target icd9 code, with edges of 3 types from corresponding graph
G_int = create_graph(icd9_codes)
for i, graph_name in enumerate(graph_names):
    with open(os.path.join(train_dir, f"{graph_name}.pkl"), "rb") as f:
        G = pickle.load(f)
    for u, v, d in G.edges(data=True):
        if G_int.has_edge(u, v):
            G_int[u][v]['weight'] += d['weight']
        else:
            G_int.add_edge(u, v, weight=d['weight'])

G_int.remove_edges_from([(u, v) for u, v, d in G_int.edges(data=True) if d['weight'] == 0]) # remove edges with weight 0
    
# save graph
with open(os.path.join(train_dir, f"sum_graph.pkl"), "wb") as f:
    pickle.dump(G_int, f)

In [ ]:
with open(os.path.join(train_dir, "sum_graph.pkl"), "rb") as f:
    G_int = pickle.load(f)

draw_graph(G_int, node_num=20)

## Build MultiGrpah


In [ ]:
# create all graph for each target icd9 code, with edges of 3 types from corresponding graph
G_multi = create_multi_graph(icd9_codes)
for i, graph_name in enumerate(graph_names):
    relation_name = graph_name.split("_")[0]
    with open(os.path.join(train_dir, f"{graph_name}.pkl"), "rb") as f:
        G = pickle.load(f)
    for u, v, d in G.edges(data=True):
        G_multi.add_edge(u, v, key=relation_name, weight=d['weight'])
    
# save graph
with open(os.path.join(train_dir, f"multi_graph.pkl"), "wb") as f:
    pickle.dump(G_multi, f)

In [ ]:
with open(os.path.join(train_dir, "multi_graph.pkl"), "rb") as f:
    G_multi = pickle.load(f)

draw_multi_graph(G_multi, node_num=20)